# Zara Sales Analysis: Commercial Data Audit & Statistical Modeling

## Overview
This notebook evaluates product performance drivers in the Kaggle Zara sales dataset through two lenses:
1. **Data Integrity Audit:** Testing whether the dataset reflects real fast-fashion retail dynamics or synthetic generation artifacts.
2. **Econometric Modeling:** Fitting a Negative Binomial GLM to evaluate price elasticity, promotional lift, and category sensitivities, along with commercial context on margin impact.

In [1]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from statsmodels.genmod.families import NegativeBinomial
from statsmodels.genmod.families.links import log
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.feature_extraction.text import TfidfVectorizer
from scipy import stats

os.makedirs('assets', exist_ok=True)
sns.set_theme(style="whitegrid", context="talk")
plt.rcParams['figure.figsize'] = (12, 7)
warnings.filterwarnings('ignore')

## 1. Data Integrity Audit
We check three basic operational dimensions to evaluate dataset authenticity:
- **Volume distribution:** Testing whether SKU volume follows the expected retail Pareto curve.
- **Merchandising taxonomy:** Checking whether store placement categories match apparel retail.
- **Origin variance:** Testing whether sales differences by sourcing country are statistically significant.

In [2]:
raw_df = pd.read_csv('Business_sales_EDA.csv', sep=';')
print(f"Loaded {len(raw_df):,} rows and {len(raw_df.columns)} columns.")

# Check 1: SKU volume distribution
vol = raw_df['Sales Volume']
print(f"Sales volume summary: min={vol.min()}, max={vol.max()}, mean={vol.mean():.1f}, std={vol.std():.1f}, skew={vol.skew():.3f}")

# Check 2: Placement categories
print("Product position breakdown:", raw_df['Product Position'].value_counts().to_dict())

# Check 3: One-way ANOVA across origin countries
origin_groups = [group['Sales Volume'].values for _, group in raw_df.groupby('origin')]
f_stat, p_val = stats.f_oneway(*origin_groups)
print(f"Origin ANOVA: F = {f_stat:.3f}, p = {p_val:.4f}")

# Visualizing audit findings
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sns.histplot(vol, kde=True, ax=axes[0], color='#2980b9')
axes[0].set_title("Sales Volume Distribution\n(Truncated bell curve)")
axes[0].set_xlabel("Sales Volume (Units)")

raw_df['Product Position'].value_counts().plot(kind='bar', ax=axes[1], color='#e67e22')
axes[1].set_title("Placement Categories\n(Grocery store taxonomy)")
axes[1].set_ylabel("SKU Count")
axes[1].tick_params(axis='x', rotation=0)

raw_df.groupby('origin')['Sales Volume'].mean().sort_values().plot(kind='barh', ax=axes[2], color='#27ae60')
axes[2].set_title(f"Mean Volume by Origin\n(ANOVA p = {p_val:.3f})")
axes[2].set_xlabel("Mean Sales Volume")
axes[2].set_xlim(1000, 1150)

plt.tight_layout()
plt.savefig('assets/forensic_data_audit.png', dpi=150)
plt.close()

Loaded 20,252 rows and 17 columns.
Sales volume summary: min=518, max=1940, mean=1097.4, std=298.2, skew=0.480
Product position breakdown: {'Aisle': 7810, 'End-cap': 6791, 'Front of Store': 5651}
Origin ANOVA: F = 1.601, p = 0.0912


## 2. Feature Engineering
Standardizing column names, logging price, and encoding indicator variables.

In [3]:
df = raw_df.rename(columns={
    'terms': 'Clothing_Type',
    'section': 'Section',
    'price': 'Price',
    'Product Position': 'Product_Position',
    'Sales Volume': 'Sales_Volume',
    'Seasonal': 'Seasonality',
    'origin': 'Origin'
})

# Normalize string formatting
df['Promotion'] = df['Promotion'].str.strip().str.title()
df['Seasonality'] = df['Seasonality'].str.strip().str.title()
df['Section'] = df['Section'].str.strip().str.upper()

# Construct modeling features
df['Log_Price'] = np.log(df['Price'] + 1)
df['Is_Promotion'] = (df['Promotion'] == 'Yes').astype(int)
df['Is_Seasonal'] = (df['Seasonality'] == 'Yes').astype(int)
df['Is_Woman'] = (df['Section'] == 'WOMAN').astype(int)
df['Near_Shoring'] = df['Origin'].isin(['Spain', 'Portugal', 'Morocco', 'Turkey']).astype(int)

print(df[['Log_Price', 'Is_Promotion', 'Is_Seasonal', 'Is_Woman', 'Near_Shoring']].head(3))

   Log_Price  Is_Promotion  Is_Seasonal  Is_Woman  Near_Shoring
0   4.381902             1            1         0             0
1   2.771964             1            0         0             1
2   4.289774             1            1         1             1


## 3. Exploratory Data Analysis
Inspecting baseline relationships across promotions, seasonality, and price.

In [4]:
# 1. Promotion volume lift
plt.figure(figsize=(8, 5))
sns.barplot(x='Promotion', y='Sales_Volume', data=df, palette=['#7f8c8d', '#2980b9'], errorbar=None)
promo_means = df.groupby('Promotion')['Sales_Volume'].mean()
lift = ((promo_means['Yes'] / promo_means['No']) - 1) * 100
plt.title(f"Promotion Lift: +{lift:.1f}% Raw Volume")
plt.ylabel("Average Sales Volume (Units)")
plt.tight_layout()
plt.savefig('assets/eda_promotion_lift.png', dpi=150)
plt.close()

# 2. Seasonality across top categories
plt.figure(figsize=(12, 5))
top_categories = df['Clothing_Type'].value_counts().head(8).index
sns.pointplot(
    x='Clothing_Type',
    y='Sales_Volume',
    hue='Seasonality',
    data=df[df['Clothing_Type'].isin(top_categories)],
    join=False,
    dodge=0.3,
    palette={'Yes': '#e74c3c', 'No': '#34495e'}
)
plt.title("Category Demand by Seasonal Flag")
plt.xticks(rotation=25)
plt.ylabel("Average Sales Volume")
plt.tight_layout()
plt.savefig('assets/eda_seasonal_patterns.png', dpi=150)
plt.close()

# 3. Price vs volume distribution
plt.figure(figsize=(9, 5))
sns.kdeplot(data=df, x='Price', y='Sales_Volume', fill=True, cmap='Blues', thresh=0.01, levels=15)
plt.title("Price Density vs. Sales Volume")
plt.xlabel("Retail Price ($)")
plt.ylabel("Sales Volume (Units)")
plt.tight_layout()
plt.savefig('assets/eda_price_distribution.png', dpi=150)
plt.close()

## 4. Text Processing & Predictor Matrix
Extracting candidate descriptors with TF-IDF while filtering common functional garment terms.

In [5]:
utility_terms = [
    'pockets', 'pocket', 'sleeves', 'sleeve', 'button', 'buttons', 'closure', 
    'front', 'back', 'interior', 'hip', 'neck', 'fit', 'hem', 'welt', 'flap', 
    'cuff', 'cuffs', 'closure', 'made', 'fabric', 'lapel', 'collar', 'long',
    'short', 'waist', 'straight', 'relaxed', 'slim', 'stretch'
]
custom_stop = list(TfidfVectorizer(stop_words='english').get_stop_words()) + utility_terms

tfidf = TfidfVectorizer(max_features=20, stop_words=custom_stop)
tfidf_matrix = tfidf.fit_transform(df['description'].fillna(''))
tfidf_cols = [f"feat_{c.replace(' ', '_')}" for c in tfidf.get_feature_names_out()]
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf_cols, index=df.index)

cat_dummies = pd.get_dummies(df['Clothing_Type'], prefix='cat', drop_first=True, dtype=int)
X_base = df[['Log_Price', 'Is_Promotion', 'Is_Seasonal', 'Is_Woman', 'Near_Shoring']]
X = pd.concat([X_base, cat_dummies, tfidf_df], axis=1)

# Seasonality interaction with top categories
for cat in cat_dummies.columns[:5]:
    X[f"Seasonal_x_{cat}"] = X['Is_Seasonal'] * X[cat]

y = df['Sales_Volume']
X_final = sm.add_constant(X.astype(float))
print(f"Design matrix shape: {X_final.shape}")

Design matrix shape: (20252, 34)


## 5. Negative Binomial GLM & Diagnostics
Fitting a Negative Binomial regression to model volume as a function of catalog and operational features.

In [6]:
nb_model = sm.GLM(y, X_final, family=NegativeBinomial(link=log())).fit()
print(f"Deviance: {nb_model.deviance:,.2f} | AIC: {nb_model.aic:,.2f} | Scale: {nb_model.scale:.4f}")

irr_df = pd.DataFrame({
    'IRR': np.exp(nb_model.params),
    'CI_Lower': np.exp(nb_model.conf_int()[0]),
    'CI_Upper': np.exp(nb_model.conf_int()[1]),
    'p_value': nb_model.pvalues
}).sort_values('IRR', ascending=False)

print("\nKey Feature Multipliers (IRRs):")
print(irr_df.drop('const', errors='ignore').head(10).to_string())

Deviance: 179.74 | AIC: 322,866.06 | Scale: 1.0000

Key Feature Multipliers (IRRs):
                           IRR  CI_Lower  CI_Upper        p_value
Is_Promotion          1.588670  1.542724  1.635985  7.636130e-210
Is_Woman              1.105788  1.074176  1.138330   1.081352e-11
feat_rib              1.010762  0.902954  1.131441   8.524377e-01
feat_round            1.008788  0.922000  1.103746   8.488170e-01
cat_jeans             1.002735  0.897240  1.120634   9.615901e-01
feat_technical        1.002324  0.891749  1.126610   9.689494e-01
Is_Seasonal           1.001933  0.964704  1.040597   9.203943e-01
Near_Shoring          1.001269  0.972453  1.030939   9.321531e-01
cat_sweaters          1.000812  0.946603  1.058126   9.771999e-01
Seasonal_x_cat_jeans  1.000723  0.855083  1.171170   9.928102e-01


In [7]:
# Residual checks
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(nb_model.fittedvalues, nb_model.resid_deviance, alpha=0.15, s=15, color='#2c3e50')
axes[0].axhline(0, color='red', linestyle='--', alpha=0.7)
axes[0].set_xlabel("Fitted Values")
axes[0].set_ylabel("Deviance Residuals")
axes[0].set_title("Fitted vs. Deviance Residuals")

stats.probplot(nb_model.resid_deviance, dist="norm", plot=axes[1])
axes[1].set_title("Q-Q Plot (Deviance Residuals)")
plt.tight_layout()
plt.close()

# Multicollinearity check on core features
base_vif = pd.DataFrame({
    'Feature': X_base.columns,
    'VIF': [variance_inflation_factor(X_base.values, i) for i in range(X_base.shape[1])]
})
print("\nBase feature VIF:")
print(base_vif.to_string(index=False))


Base feature VIF:
     Feature      VIF
   Log_Price 4.296963
Is_Promotion 1.805925
 Is_Seasonal 2.128865
    Is_Woman 2.795205
Near_Shoring 1.495789


## 6. Structural Sales Drivers (IRR Map)
Visualizing the top multipliers with 95% confidence intervals.

In [8]:
plot_drivers = irr_df.drop('const', errors='ignore').head(15).iloc[::-1]

plt.figure(figsize=(11, 7))
xerr = [
    plot_drivers['IRR'] - plot_drivers['CI_Lower'],
    plot_drivers['CI_Upper'] - plot_drivers['IRR']
]
colors = ['#27ae60' if v >= 1.0 else '#e74c3c' for v in plot_drivers['IRR']]

plt.barh(plot_drivers.index, plot_drivers['IRR'], color=colors, alpha=0.85, xerr=xerr, capsize=3)
plt.axvline(1.0, color='black', linestyle='--', alpha=0.7, label='Baseline (IRR = 1.0)')
plt.xlabel("Incident Rate Ratio (Volume Multiplier with 95% CI)")
plt.title("Estimated Sales Multipliers (Negative Binomial GLM)")
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('assets/strategic_sales_drivers.png', dpi=150)
plt.close()

## 7. Category-Level Sensitivity Analysis
Evaluating price sensitivity and promotional lift independently by clothing category.

In [9]:
top_categories_list = df['Clothing_Type'].value_counts().head(6).index
df_top = df[df['Clothing_Type'].isin(top_categories_list)].copy()

# Visual 1: Price regressions by category
g = sns.lmplot(
    data=df_top,
    x="Price",
    y="Sales_Volume",
    col="Clothing_Type",
    col_wrap=3,
    height=3.5,
    aspect=1.2,
    scatter_kws={'alpha': 0.08, 'color': '#2c3e50'},
    line_kws={'color': '#e74c3c'}
)
g.set_axis_labels("Price ($)", "Sales Volume")
g.fig.subplots_adjust(top=0.9)
g.fig.suptitle("Price Sensitivity Across Top Categories")
plt.savefig('assets/segmented_price_sensitivity.png', dpi=150)
plt.close()

# Visual 2: Category promotional lift
plt.figure(figsize=(11, 5))
sns.barplot(x='Clothing_Type', y='Sales_Volume', hue='Promotion', data=df_top, palette=['#95a5a6', '#2980b9'], errorbar=None)
plt.title("Promotional Volume Lift by Category")
plt.ylabel("Average Sales Volume")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig('assets/segmented_promotion_lift.png', dpi=150)
plt.close()

# Fit category-level models
cat_summary = []
for category in df['Clothing_Type'].unique():
    sub = df[df['Clothing_Type'] == category].copy()
    if len(sub) < 150:
        continue
    sub_X = sm.add_constant(sub[['Log_Price', 'Is_Promotion', 'Is_Seasonal', 'Is_Woman']].astype(float))
    sub_y = sub['Sales_Volume']
    try:
        m = sm.GLM(sub_y, sub_X, family=NegativeBinomial(link=log())).fit()
        m_irr = np.exp(m.params)
        cat_summary.append({
            'Category': category,
            'Sample_Size': len(sub),
            'Price_Elasticity_IRR': m_irr['Log_Price'],
            'Promo_Lift_IRR': m_irr['Is_Promotion'],
            'Seasonal_Lift_IRR': m_irr['Is_Seasonal']
        })
    except Exception:
        continue

summary_df = pd.DataFrame(cat_summary).set_index('Category').sort_values('Price_Elasticity_IRR')
print("\nCategory Sensitivity Matrix (IRRs):")
print(summary_df.to_string())

# Visual 3: Comparative category sensitivity
plt.figure(figsize=(10, 6))
summary_df[['Price_Elasticity_IRR', 'Promo_Lift_IRR']].plot(kind='barh', ax=plt.gca(), width=0.8, color=['#e74c3c', '#27ae60'])
plt.axvline(1.0, color='black', linestyle='--', alpha=0.6, label='Neutral Effect (1.0)')
plt.title("Category Sensitivity: Price Elasticity vs. Promotion Lift")
plt.xlabel("Incident Rate Ratio (Multiplier)")
plt.legend(loc='lower right')
plt.tight_layout()
plt.savefig('assets/comparative_segmented_analysis.png', dpi=150)
plt.close()


Category Sensitivity Matrix (IRRs):
          Sample_Size  Price_Elasticity_IRR  Promo_Lift_IRR  Seasonal_Lift_IRR
Category                                                                      
shoes            2458              0.886102        1.583873           1.004109
t-shirts         2646              0.887232        1.588163           1.000563
jackets         11232              0.887510        1.589491           1.001683
sweaters         3257              0.892415        1.589166           1.003070
jeans             659              0.895416        1.601689           0.998183


## 8. Summary of Commercial Implications
1. **Dataset Caveats:** The synthetic distribution, grocery shelf placement variables, and origin uniformity indicate that this dataset cannot be used for operational decision-making.
2. **Margin Considerations:** While promotions generate a ~1.59x unit volume lift, evaluating this in isolation without cost of goods sold and return rates risks destroying profitability.
3. **Production Requirements:** Practical retail analytics requires weekly store-level transactions, landed unit costs, inventory stockout metrics, and sell-through percentages.